# 06 — End-to-end: same call the SEC page makes

`sec_service.run_extraction(sample_name, use_cache=..., excel_out=...)`
ties everything together: cache load → analyzer deploy → classify+extract
(with retries) → merge → Excel export → projection to `SecExtractionResult`.

This is the canonical "what the app does" reference. When the SEC page's
**Run extraction** button completes, the result returned is exactly what
this notebook prints.


In [ ]:
from _lib import sec_service
from pathlib import Path

sample = sec_service.list_samples()[0]['file_name']
excel_out = sec_service.SEC_DEMO_ROOT / 'output' / f'{Path(sample).stem}__nb06.xlsx'
result = sec_service.run_extraction(
    sample, use_cache=True, excel_out=excel_out
)

print('file_name        :', result['file_name'])
print('from_cache       :', result['meta'].get('from_cache'))
print('elapsed_sec      :', result['meta'].get('elapsed_sec'))
print('retries          :', result['meta'].get('retries'))
print('segment counts   :', result['meta'].get('segment_categories'))
print('missing statements:', result['meta'].get('missing_statements'))
print('statements       :', [s['statement_type'] for s in result['statements']])
print('xlsx artifact    :', result['meta'].get('artifacts', {}).get('excel'))

## Parity assertion

The API's `/sec/extract` endpoint coerces this same dict into
`SecExtractionResult` (Pydantic). The fields below are the contract the
frontend's `SecExtractionResult` TS type expects.

In [ ]:
expected_keys = {'file_name', 'meta', 'statements', 'raw'}
assert expected_keys.issubset(result.keys()), result.keys()
for s in result['statements']:
    assert {'statement_type', 'rows', 'period_headers'}.issubset(s.keys())
print('OK — shape matches SecExtractionResult contract')